# 00e — Generate Symile PULSE candidates

Use your **working PULSE Python 3.9.15 environment** on a GPU allocation. Do not install the graph project's requirements into this kernel. Requires `candidate_preparation.py` and `sensor_candidate_generation.py` beside this notebook.

This runs actual sensor-only inference; no retrieved examples or reference annotations enter the prompt. The selected sensor tensors are read using 00b source indices and checked against native admission IDs. Default is a **10-record smoke run**. After inspecting the output quality, set `MAX_NEW_RECORDS = 0` to process all remaining records. Reruns skip successful records. Set `RETRY_ERRORS = True` to retry failed records under the same protocol. Changing model/prompt/preprocessing requires a new generation directory.

The input is the released Symile preprocessed tensor. CXR pixels are reconstructed by inverse ImageNet normalization; ECGs are rendered as 12 normalized-amplitude leads at 500 Hz. These are not pristine original sensor files. ECG voltage/amplitude diagnoses cannot be validated from normalized signals. Confirm lead order against the source preprocessing and visually inspect representative rendered images before a full run. This renderer follows the 12-lead layout of Experiment 4, but uses normalized Symile signals.

Actual GPU inference has not been tested locally. The model and processor are loaded from your existing checkpoint; there is no package upgrade or explicit checkpoint download. The PULSE backend may additionally need its cached vision-tower weights.

In [ ]:
from pathlib import Path
import json, sys
NOTEBOOK_DIR=Path.cwd().resolve()
PROJECT_ROOT=next((p for p in (NOTEBOOK_DIR,*NOTEBOOK_DIR.parents) if (p/'config.json').is_file()),None)
if PROJECT_ROOT is None:raise FileNotFoundError('Start in graph_reasoning or its extraction subfolder; project config.json is required')
HELPER_ROOT=next((p for p in (NOTEBOOK_DIR,PROJECT_ROOT,PROJECT_ROOT/'symile_extraction_update') if (p/'candidate_preparation.py').is_file() and (p/'sensor_candidate_generation.py').is_file()),None)
if HELPER_ROOT is None:raise FileNotFoundError('Copy both preparation helpers into the project or extraction subfolder')
sys.path.insert(0,str(HELPER_ROOT))
from candidate_preparation import *
CONFIG=read(PROJECT_ROOT/'config.json')
def resolve(value):
    p=Path(value).expanduser()
    return (p if p.is_absolute() else PROJECT_ROOT/p).resolve()
SOURCES=CONFIG.get('export_sources',{})
MANIFEST=resolve(SOURCES.get('manifest','data/prepared/manifest.json'))
SENSORS=MANIFEST.parent/'sensor_sources.json'
ARTIFACT_ROOT=resolve(CONFIG.get('embedding_artifact_root','/path/to/data/symile/ecg_cxr_bridge_artifacts'))
GENERATION_DIR=resolve(CONFIG.get('candidate_generation_dir','outputs/symile_sensor_only_v1'))
assert MANIFEST.is_file() and SENSORS.is_file(), 'Run notebook 00b first'


In [ ]:
MODALITY='ecg'
from sensor_candidate_generation import sensor_plan, find_snapshot, load_runtime, run_generation
SYMILE_ROOT=resolve(CONFIG.get('symile_root','/path/to/data/symile/physionet.org/files/symile-mimic/1.0.0'))
# Optional: set this to the local checkpoint directory used by your successful Experiment 4 run.
MODEL_PATH=None
import os
PULSE_REPO=Path(os.environ.get('PULSE_REPO_ROOT','/path/to/data/models/PULSE'))
MAX_NEW_RECORDS=10  # After a successful smoke run, use 0 for all remaining records.
RETRY_ERRORS=False
MAX_NEW_TOKENS=384
# Leave the directory identical between 00e and 00f so 00g can combine the outputs.
print('Kernel:',sys.executable)
print('Generation directory:',GENERATION_DIR)
tasks,arrays=sensor_plan(MANIFEST,SENSORS,SYMILE_ROOT,MODALITY)
print('Selected sensor records:',len(tasks))
MODEL_PATH=find_snapshot(MODELS[MODALITY],MODEL_PATH)
print('Using existing model snapshot:',MODEL_PATH)


In [ ]:
runtime=load_runtime(MODALITY,MODEL_PATH,pulse_repo=PULSE_REPO if MODALITY=='ecg' else None)
summary=run_generation(tasks,arrays,runtime,GENERATION_DIR,limit=MAX_NEW_RECORDS,
                       retry_errors=RETRY_ERRORS,max_new_tokens=MAX_NEW_TOKENS)
print(summary)
print('Reports:',GENERATION_DIR/f'{MODALITY}_sensor_only.jsonl')
print('Complete all records (ok == total) before 00g. Review failures in the JSONL; no failed case is silently removed.')
